# Possible Improvement 1: Query Expansion via Synonyms or Word Embeddings

# findings: actually made the MAP@K lower to 0.18 from 0.29

# possible reasons:

It generates all synonyms, homonyms, or unrelated lemmas, which dilutes your query.

For example, "armchair" might get synonyms like "president" (from politics) — hurting precision.

In [14]:
import sys
import os
import json 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ENV = 'dev'

# Load config
with open("config.json") as f:
    config = json.load(f)

if ENV == 'dev':
    base_path = config[f"{ENV}_path"]  
    data_path = os.path.join(base_path, "data")
    model_path = os.path.join(base_path, "models")
    print("Base path:", base_path)    
    print("Data path:", data_path)
    print("Model path:", model_path)

Base path: /Users/jillchow/HBS/hbs_search_engine
Data path: /Users/jillchow/HBS/hbs_search_engine/data
Model path: /Users/jillchow/HBS/hbs_search_engine/models


In [2]:
queryfile_name = "query.csv" 
queryfile_path = os.path.join(data_path, queryfile_name)
productfile_name = "product.csv" 
productfile_path = os.path.join(data_path, productfile_name)
labelfile_name = "label.csv" 
labelfile_path = os.path.join(data_path, labelfile_name)


query_df = pd.read_csv(queryfile_path, sep='\t')
product_df = pd.read_csv(productfile_path, sep='\t')
label_df = pd.read_csv(labelfile_path, sep='\t')

print('query_df: search queries')
display(query_df.head()) # watch for null values in query class column 
query_df.info()

print('\n product_df: product information')
display(product_df.head())
product_df.info() # watch for null values other than product_id, product_name, product_features

print('\n label_df: ground truth labels')
display(label_df.head())
label_df.info()

query_df: search queries


,query_id,query,query_class
0,0,salon chair,Massage Chairs
1,1,smart coffee table,Coffee & Cocktail Tables
2,2,dinosaur,Kids Wall Décor
3,3,turquoise pillows,Accent Pillows
4,4,chair and a half recliner,Recliners


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   query_id     480 non-null    int64 
 1   query        480 non-null    object
 2   query_class  474 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.4+ KB

 product_df: product information


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42994 entries, 0 to 42993
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           42994 non-null  int64  
 1   product_name         42994 non-null  object 
 2   product_class        40142 non-null  object 
 3   category hierarchy   41438 non-null  object 
 4   product_description  36986 non-null  object 
 5   product_features     42994 non-null  object 
 6   rating_count         33542 non-null  float64
 7   average_rating       33542 non-null  float64
 8   review_count         33542 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.0+ MB

 label_df: ground truth labels


,id,query_id,product_id,label
0,0,0,25434,Exact
1,1,0,12088,Irrelevant
2,2,0,42931,Exact
3,3,0,2636,Exact
4,4,0,42923,Exact


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233448 entries, 0 to 233447
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          233448 non-null  int64 
 1   query_id    233448 non-null  int64 
 2   product_id  233448 non-null  int64 
 3   label       233448 non-null  object
dtypes: int64(3), object(1)
memory usage: 7.1+ MB


In [20]:
import importlib
import helper
importlib.reload(helper)
from helper import calculate_tfidf, get_top_products, map_at_k, get_top_product_ids_for_query, get_exact_matches_for_query

In [31]:
import nltk
from nltk.corpus import wordnet

# nltk.download('wordnet')
# nltk.download('omw-1.4')

def expand_query(query):
    words = query.split()
    expanded = set(words)
    for word in words:
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                expanded.add(lemma.name().replace('_', ' '))
    return ' '.join(expanded)

def get_top_products_expanded(vectorizer, tfidf_matrix, query, top_n=10):
    expanded_query = expand_query(query)
    query_vector = vectorizer.transform([expanded_query])
    cosine_similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_product_indices = cosine_similarities.argsort()[-top_n:][::-1]
    return top_product_indices


# update to use the expanded function above
# also pass dependencies explicitly
def get_top_product_ids_for_query(query, vectorizer, tfidf_matrix, product_df):
    top_product_indices = get_top_products_expanded(vectorizer, tfidf_matrix, query, top_n=10)
    top_product_ids = product_df.iloc[top_product_indices]['product_id'].tolist()
    return top_product_ids

# update to use the label_df, and add to the parameters
def get_exact_matches_for_query(query_id, label_df):
    grouped_label_df = label_df.groupby('query_id')
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches

In [38]:
# Calculate TF-IDF
vectorizer, tfidf_matrix = calculate_tfidf(product_df)

# applying the function to obtain top product IDs and adding top K product IDs to the dataframe 

query_df['top_product_ids'] = query_df['query'].apply(
    lambda q: get_top_product_ids_for_query(q, vectorizer, tfidf_matrix, product_df)
)
# adding the list of exact match product_IDs from labels_df
query_df['relevant_ids'] = query_df['query_id'].apply(
      lambda qid: get_exact_matches_for_query(qid, label_df)
)

# now assign the map@k score
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)

query_df.loc[:, 'map@k'].mean()

0.187511257164903

In [ ]:
query_df.head(10)

,query_id,query,query_class,top_product_ids,relevant_ids,map@k
0,0,salon chair,Massage Chairs,"[2187, 7466, 7467, 15612, 7468, 7465, 7506, 29...","[25434, 42931, 2636, 42923, 41156, 5936, 22390...",0.430000
1,1,smart coffee table,Coffee & Cocktail Tables,"[28095, 28096, 19926, 34786, 19116, 23575, 268...","[9929, 5235, 37304, 25973, 16679, 29449, 33698...",0.000000
2,2,dinosaur,Kids Wall Décor,"[34737, 24094, 14418, 34735, 34736, 34739, 375...","[4205, 4202, 4204, 36622, 29777, 40289, 10539,...",1.000000
3,3,turquoise pillows,Accent Pillows,"[21037, 109, 25672, 26670, 41503, 41499, 4675,...","[18909, 12386, 12436, 32704, 12201, 18293, 404...",0.000000
4,4,chair and a half recliner,Recliners,"[22744, 6519, 34103, 19671, 2185, 1589, 21190,...","[5488, 6098, 42393, 16598, 41662, 40331, 24881...",0.000000
5,5,sofa with ottoman,Sectionals,"[33252, 22541, 40072, 33251, 5073, 36365, 1739...","[33253, 19757, 26053, 6986, 25713, 14091, 4139...",0.241667
6,6,acrylic clear chair,Dining Chairs,"[25143, 26875, 19248, 26876, 32575, 34110, 529...","[41828, 32573, 25711, 1454, 28922, 25147, 1981...",0.226667
7,7,driftwood mirror,Wall & Accent Mirrors,"[11297, 34926, 34083, 37655, 771, 37652, 37651...","[37651, 19768, 27648, 37652, 37645, 34787, 142...",0.736548
8,8,home sweet home sign,Wall Décor,"[11511, 30355, 28089, 30358, 3728, 21994, 2127...","[30082, 21994, 19132, 23011, 22077, 889, 8301,...",0.569048
9,9,coffee table fire pit,Outdoor Fireplaces,"[7507, 7509, 8715, 21800, 38098, 7997, 29691, ...","[20907, 30092, 3288, 29691, 30826, 13615, 2936...",0.900000
